# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HemapavaniDontula/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

### Method choice — Random Forest Regression

I use a Random Forest regression model to study whether the available
content and search signals can provide useful directional information
about observed trend severity.

I define decline severity as the negative of the observed `trend_pct`,
so higher values represent more negative observed movement. This makes
the model ranking comparable with the Week-4 baseline investigation score.

Random Forest can capture non-linear relationships and interactions
without assuming a linear relationship. The model will be compared with
the Week-4 baseline on the same validation split and using the same
ranking metric.

The result is treated as observed and directional evidence for
decision-support, not causal evidence about search ranking or content
changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a grouped train-validation split by `client_id`.

The dataset contains multiple observations associated with clients, so
grouping by client helps prevent observations from the same client from
appearing in both training and validation sets.

I use 80% of the client groups for training and 20% for validation, with
a fixed random seed for reproducibility.

This split is used for both the model and the Week-4 baseline comparison.
The validation set is kept separate when fitting model parameters and
baseline thresholds.

In [20]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

repo_path = Path("/content/flyrank-ml")

if not repo_path.exists():
    !git clone --depth 1 https://github.com/HemapavaniDontula/flyrank-ml.git /content/flyrank-ml

os.chdir(repo_path)

data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Current directory:", os.getcwd())
print("Shape:", df.shape)
print("Columns:", len(df.columns))

Dataset loaded successfully
Current directory: /content/flyrank-ml
Shape: (30000, 44)
Columns: 44


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(
        df,
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("Train clients:", train_df["client_id"].nunique())
print("Validation clients:", val_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(val_df["client_id"])
print("Client overlap:", len(overlap))

Train shape: (23837, 44)
Validation shape: (6163, 44)
Train clients: 25
Validation clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

The target is observed decline severity, defined as `-trend_pct`.
Higher values therefore represent more negative observed movement.

The model is trained only on the training groups and evaluated on the
held-out validation groups.

For a fair ranking comparison, I use Spearman rank correlation between
the investigation ranking and observed decline severity.

The Week-4 baseline assigns higher scores to observations with stronger
staleness and CTR-related investigation signals. The model and baseline
are therefore compared on the same validation observations and the same
ranking metric.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

train_df["decline_severity"] = -pd.to_numeric(
    train_df["trend_pct"],
    errors="coerce"
)

val_df["decline_severity"] = -pd.to_numeric(
    val_df["trend_pct"],
    errors="coerce"
)

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

Training rows: 23837
Validation rows: 6163


In [8]:
feature_cols = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "position_tier",
    "avg_position",
    "ctr"
]

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

y_train = train_df["decline_severity"].copy()
y_val = val_df["decline_severity"].copy()

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "position_tier"
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'avg_position', 'ctr']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'position_tier']


In [10]:
# Target: observed decline severity
# Higher value = more negative observed trend

train_df["decline_severity"] = -pd.to_numeric(
    train_df["trend_pct"],
    errors="coerce"
)

val_df["decline_severity"] = -pd.to_numeric(
    val_df["trend_pct"],
    errors="coerce"
)

# Remove rows where the target is missing
train_df = train_df.dropna(
    subset=["decline_severity"]
).copy()

val_df = val_df.dropna(
    subset=["decline_severity"]
).copy()

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

print(
    "Missing training targets:",
    train_df["decline_severity"].isna().sum()
)

print(
    "Missing validation targets:",
    val_df["decline_severity"].isna().sum()
)

Training rows: 21133
Validation rows: 5479
Missing training targets: 0
Missing validation targets: 0


In [11]:
baseline_train = train_df.copy()
baseline_val = val_df.copy()

# Learn baseline thresholds only from training data
stale_cutoff = baseline_train["days_since_last_update"].quantile(2/3)

position_ctr_median = (
    baseline_train
    .groupby("position_tier")["ctr"]
    .median()
)

# Apply learned rules to validation data
baseline_val["_stale_signal"] = (
    baseline_val["days_since_last_update"] >= stale_cutoff
).astype(int)

baseline_val["_position_ctr_median"] = (
    baseline_val["position_tier"]
    .map(position_ctr_median)
)

baseline_val["_ctr_signal"] = (
    baseline_val["ctr"] < baseline_val["_position_ctr_median"]
).astype(int)

baseline_val["_position_signal"] = (
    baseline_val["position_tier"].isin(
        ["top_3", "page_1"]
    )
).astype(int)

baseline_val["baseline_score"] = (
    2 * baseline_val["_stale_signal"]
    + 2 * baseline_val["_ctr_signal"]
    + baseline_val["_position_signal"]
)

print(
    baseline_val[
        [
            "baseline_score",
            "_stale_signal",
            "_ctr_signal",
            "_position_signal"
        ]
    ].head(10)
)

    baseline_score  _stale_signal  _ctr_signal  _position_signal
0                0              0            0                 0
1                0              0            0                 0
5                3              0            1                 1
13               2              0            1                 0
19               1              0            0                 1
20               1              0            0                 1
22               3              0            1                 1
23               0              0            0                 0
25               3              0            1                 1
26               0              0            0                 0


In [15]:
comparison_df = pd.DataFrame({
    "actual_decline_severity": y_val.values,
    "baseline_score": baseline_val["baseline_score"].values,
    "model_prediction": model_pred
})

baseline_spearman = spearmanr(
    comparison_df["baseline_score"],
    comparison_df["actual_decline_severity"]
).statistic

model_spearman = spearmanr(
    comparison_df["model_prediction"],
    comparison_df["actual_decline_severity"]
).statistic

comparison_table = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Spearman_rank_correlation": [
        baseline_spearman,
        model_spearman
    ]
})

print(comparison_table.to_string(index=False))

         Method  Spearman_rank_correlation
Week-4 baseline                   0.186973
  Random Forest                   0.151247


In [14]:
rf_pipeline.fit(X_train, y_train)

model_pred = rf_pipeline.predict(X_val)

print("Model trained successfully.")
print("Predictions:", len(model_pred))

Model trained successfully.
Predictions: 5479


In [13]:
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

y_train = train_df["decline_severity"].copy()
y_val = val_df["decline_severity"].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("NaN in y_train:", y_train.isna().sum())
print("NaN in y_val:", y_val.isna().sum())

X_train: (21133, 17)
y_train: (21133,)
X_val: (5479, 17)
y_val: (5479,)
NaN in y_train: 0
NaN in y_val: 0


In [16]:
from sklearn.metrics import mean_absolute_error

model_mae = mean_absolute_error(
    y_val,
    model_pred
)

print("Random Forest validation MAE:", model_mae)

Random Forest validation MAE: 82.20962347786596


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The model is evaluated using its validation predictions rather than the
training data.

Large absolute prediction errors identify observations where the model
has difficulty estimating observed decline severity.

Feature importance is used only as an interpretation aid. It shows which
available signals the Random Forest relied on most strongly; it does not
show that those signals caused the observed trend.

The findings are treated as directional and suitable for decision-support,
not as causal evidence.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
error_df = val_df[
    [
        "content_id",
        "trend_pct",
        "decline_severity"
    ]
].copy()

error_df["prediction"] = model_pred

error_df["absolute_error"] = (
    error_df["decline_severity"]
    - error_df["prediction"]
).abs()

largest_errors = (
    error_df
    .sort_values("absolute_error", ascending=False)
    .head(10)
)

print(largest_errors.to_string(index=False))

          content_id  trend_pct  decline_severity   prediction  absolute_error
content_22f4d2f58c42    21400.0          -21400.0    53.586923    21453.586923
content_32ff84795595    14900.0          -14900.0     9.170469    14909.170469
content_f560a011a094     8400.0           -8400.0    38.388992     8438.388992
content_94feac677ec4     6212.5           -6212.5    11.871926     6224.371926
content_a869de96aa65     6000.0           -6000.0    30.527676     6030.527676
content_84bc4706ba53      -88.1              88.1 -3936.493448     4024.593448
content_147608c766c3      -76.9              76.9 -3925.273050     4002.173050
content_f8d0a67e22fc     4020.0           -4020.0   -30.929468     3989.070532
content_5b377954b9b8        0.0              -0.0 -3801.338124     3801.338124
content_5723e05acdc6      -96.7              96.7 -3687.271487     3783.971487


In [18]:
feature_names = (
    rf_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

importances = (
    rf_pipeline
    .named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

print(
    importance_df.head(15).to_string(index=False)
)

                       feature  importance
         num__content_age_days    0.329420
             num__avg_position    0.311373
              num__competition    0.095315
                      num__cpc    0.053187
               num__char_count    0.046829
                      num__ctr    0.042802
               num__word_count    0.040713
            num__search_volume    0.034587
           num__age_tier_order    0.011291
   cat__main_intent_commercial    0.007499
   num__days_since_last_update    0.007167
cat__main_intent_transactional    0.004765
   cat__position_tier_striking    0.002158
 cat__competition_level_MEDIUM    0.002125
      cat__position_tier_top_3    0.001866


### Interpretation

The validation comparison shows how the Random Forest ranking performs
relative to the Week-4 rule-based baseline on the same held-out groups.

The feature importance results identify which available signals receive
more weight from the fitted model. These should be interpreted as
predictive signals within this dataset, not causal drivers.

Large-error observations are useful for identifying cases where the model
is less reliable and where additional investigation may be needed.

The model should therefore be used as decision-support rather than as a
guaranteed rule for content changes or search-ranking outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.